# Chapter 13 — Deploying scikit-learn Models in Production

**Nama:** Omar Al Farizy Hutagalung  

**NIM:** 101032300034  

**Mata Kuliah:** Machine Learning  

**Tugas:** Tugas 2 — Code Reproduction & Theoretical Deep-Dive

---

## Objective

Chapter ini membahas persiapan model untuk produksi, penyimpanan dan pemuatan model, menjaga konsistensi preprocessing, serta dasar penyajian prediksi melalui API sederhana.

## 1. Landasan Teori

Deployment mengubah model hasil eksperimen menjadi komponen yang dapat digunakan sistem lain.

Prinsip utama:
- Simpan preprocessing bersama model menggunakan Pipeline.
- Pastikan urutan, nama, tipe, dan skema fitur saat inferensi sama dengan saat training.
- Evaluasi model pada data yang representatif sebelum deployment.
- Catat versi model, dependensi, metrik, dan sumber data.
- Lindungi endpoint dari input tidak valid dan akses yang tidak sah.
- Jangan memuat file model dari sumber yang tidak dipercaya.

`joblib` praktis untuk menyimpan objek Python/scikit-learn, tetapi file pickle/joblib dapat menjalankan kode saat dimuat. Hanya muat artefak tepercaya dan perhatikan kompatibilitas versi library.

In [ ]:
import numpy as np
import pandas as pd
import sklearn

print("scikit-learn:", sklearn.__version__)

## 2. Melatih dan menyimpan Pipeline

Contoh ini melatih model klasifikasi, menyimpan Pipeline, lalu memuatnya kembali. Karena scaler dan classifier berada dalam satu Pipeline, keduanya ikut tersimpan.

In [ ]:
from pathlib import Path
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=42)
)
model.fit(X_train, y_train)

artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
model_path = artifact_dir / "breast_cancer_pipeline.joblib"

joblib.dump(model, model_path)
print("Model disimpan ke:", model_path.resolve())
print("Test accuracy:", accuracy_score(y_test, model.predict(X_test)))

## 3. Memuat kembali dan menjalankan inferensi

In [ ]:
loaded_model = joblib.load(model_path)

sample = X_test[:3]
predictions = loaded_model.predict(sample)
probabilities = loaded_model.predict_proba(sample)

print("Prediksi:", predictions)
print("Probabilitas per kelas:")
print(np.round(probabilities, 3))

assert np.array_equal(model.predict(sample), loaded_model.predict(sample))
print("Verifikasi: prediksi model asli dan model yang dimuat kembali sama.")

## 4. Menjaga kontrak input dan skema fitur

Pada dataset ini, fitur berupa array dengan urutan kolom yang ditentukan dataset. Pada aplikasi produksi, gunakan nama kolom dan validasi skema secara eksplisit jika input berasal dari JSON atau sistem eksternal.

In [ ]:
from sklearn.datasets import load_breast_cancer

dataset = load_breast_cancer()
feature_names = list(dataset.feature_names)

example_payload = {
    name: float(value)
    for name, value in zip(feature_names, X_test[0])
}

print("Jumlah fitur yang diharapkan:", len(feature_names))
print("Jumlah fitur pada contoh payload:", len(example_payload))
print("Contoh beberapa field:", list(example_payload.items())[:5])

def validate_payload(payload, expected_features):
    missing = set(expected_features) - set(payload)
    extra = set(payload) - set(expected_features)
    if missing or extra:
        raise ValueError(
            f"Skema input tidak sesuai. Missing={sorted(missing)}, "
            f"extra={sorted(extra)}"
        )
    values = np.array([[payload[name] for name in expected_features]], dtype=float)
    if not np.isfinite(values).all():
        raise ValueError("Semua nilai fitur harus berupa angka finite.")
    return values

validated_sample = validate_payload(example_payload, feature_names)
print("Prediksi dari payload tervalidasi:", loaded_model.predict(validated_sample))

## 5. Contoh API dengan FastAPI (opsional)

Contoh ini memperlihatkan bentuk minimal endpoint prediksi. Jalankan di lingkungan aplikasi terpisah setelah menginstal FastAPI dan Uvicorn. Jangan membuka endpoint ke jaringan publik tanpa autentikasi, validasi input, dan konfigurasi keamanan yang sesuai.

Simpan kode berikut sebagai `app.py` di lokasi yang dapat mengakses folder `artifacts/`.

In [ ]:
# app.py — contoh terpisah, bukan dijalankan otomatis di notebook
#
# from fastapi import FastAPI, HTTPException
# from pydantic import BaseModel
# import joblib
# import numpy as np
# from pathlib import Path
#
# app = FastAPI(title="scikit-learn Prediction API")
# model = joblib.load(Path("artifacts/breast_cancer_pipeline.joblib"))
#
# class PredictionRequest(BaseModel):
#     features: list[float]
#
# @app.get("/health")
# def health():
#     return {"status": "ok"}
#
# @app.post("/predict")
# def predict(request: PredictionRequest):
#     expected = model.n_features_in_
#     if len(request.features) != expected:
#         raise HTTPException(
#             status_code=422,
#             detail=f"Expected {expected} features"
#         )
#     values = np.asarray(request.features, dtype=float).reshape(1, -1)
#     if not np.isfinite(values).all():
#         raise HTTPException(status_code=422, detail="Features must be finite")
#     prediction = int(model.predict(values)[0])
#     return {"prediction": prediction}
#
# Jalankan dari terminal:
# pip install fastapi uvicorn
# uvicorn app:app --host 127.0.0.1 --port 8000
#
# Gunakan host 127.0.0.1 untuk pengujian lokal.

## 6. Checklist produksi

- [ ] Validasi input dan skema fitur.
- [ ] Simpan Pipeline, bukan hanya classifier terpisah.
- [ ] Catat versi Python dan scikit-learn yang digunakan.
- [ ] Uji model setelah artefak dimuat ulang.
- [ ] Lindungi artefak model dan endpoint.
- [ ] Pantau error, latensi, distribusi input, dan perubahan performa.
- [ ] Siapkan proses versioning dan rollback.

## 7. Analisis dan Kesimpulan

Deployment bukan hanya menyimpan file model. Sistem produksi juga memerlukan kontrak input, validasi, keamanan, pencatatan versi, monitoring, dan proses pembaruan.

**Kesimpulan:** Pipeline dan artefak yang terversi membantu menjaga konsistensi antara training dan inference. Contoh ini merupakan demonstrasi lokal, bukan layanan produksi yang sudah diamankan atau diuji pada beban nyata.